# 08 — Document Loaders

Load documents from text files, CSV, and web pages, then split them.

In [ ]:
import os, csv, tempfile
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, CSVLoader, WebBaseLoader

## Example 1: Text Loader

In [ ]:
with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
    f.write("LangChain is a framework for building LLM applications.\n"
            "It provides abstractions for chains, agents, and retrieval.\n"
            "LangChain supports multiple language model providers.\n"
            "The framework is available in Python and JavaScript.")
    tmp_path = f.name

loader = TextLoader(tmp_path)
text_docs = loader.load()
print(f"Loaded {len(text_docs)} document(s)")
print(f"Preview: {text_docs[0].page_content[:80]}...")
os.unlink(tmp_path)

## Example 2: CSV Loader

In [ ]:
with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", delete=False, newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["name", "language", "stars", "description"])
    writer.writerow(["LangChain", "Python", "75000", "Framework for LLM apps"])
    writer.writerow(["LlamaIndex", "Python", "30000", "Data framework for LLMs"])
    writer.writerow(["Semantic Kernel", "C#", "18000", "Microsoft LLM orchestration"])
    tmp_path = f.name

loader = CSVLoader(tmp_path)
csv_docs = loader.load()
print(f"Loaded {len(csv_docs)} rows as documents")
for doc in csv_docs[:2]:
    print(f"  Row: {doc.page_content[:60]}...")
os.unlink(tmp_path)

## Example 3: Web Loader

In [ ]:
try:
    loader = WebBaseLoader("https://en.wikipedia.org/wiki/Large_language_model")
    web_docs = loader.load()
    print(f"Loaded {len(web_docs)} document(s) from web")
    print(f"Content length: {len(web_docs[0].page_content)} characters")
except Exception as e:
    print(f"Web loading skipped: {e}")
    web_docs = [Document(page_content="Large language models (LLMs) are AI models trained on vast text data.", metadata={"source": "fallback"})]

## Combined Pipeline with Splitting

In [ ]:
all_docs = text_docs + csv_docs + web_docs
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = splitter.split_documents(all_docs)
print(f"Total documents: {len(all_docs)} → {len(chunks)} chunks")
sizes = [len(c.page_content) for c in chunks]
print(f"Chunk sizes: min={min(sizes)}, max={max(sizes)}, avg={sum(sizes)//len(sizes)}")